# DFXM dislocation identification → labeled ML training data

*A walkthrough of the modernized `dfxm-geo` simulation pipeline, focused on
generating many dislocation configurations as labeled training data.*

You already know the DFXM physics and the **original Geometrical-Optics
scripts**. This notebook is a tour of what that code has become since the
cleanup/modernization, and of the part that is genuinely new for you: a
**labeled, multi-dislocation image generator** built for machine learning,
plus a path to scale it to 100k+ images.

The notebook **runs end to end** — every image and label table below is
produced live by the pipeline on this machine.

## 1 · What changed since the old code

| | **Then** (Geometrical_Optics scripts) | **Now** (`dfxm-geo`) |
|---|---|---|
| Use | edit globals in `functions.py` / `image_processor.py`, run a script | `pip install dfxm-geo`; two CLIs + TOML configs |
| Entry points | hand-run modules | **`dfxm-forward`** (forward sim) · **`dfxm-identify`** (labeled datasets) · `dfxm-bootstrap` (resolution kernel) |
| Output | `.npy` dumps + ad-hoc filenames | **HDF5 / NeXus** in the BLISS layout (silx/darfix/darling-readable) |
| Speed | pure NumPy | `@njit` (numba) fused kernels — the per-population strain solve is ~15× faster |
| Distribution | a folder you copied | on **PyPI** and **conda-forge**, versioned + tested |

The physics core is the same model you knew. What's new for *your* problem is
`dfxm-identify --mode multi`: it draws random dislocations, renders the image,
**and writes the ground-truth labels next to it**.

## 2 · `dfxm-forward` vs `dfxm-identify`

Both share the same forward physics: dislocation → displacement-gradient field
$H_g$ → convolution with the bootstrapped reciprocal-space resolution kernel →
detector image. They differ in **what they loop over** and **what they record**.

- **`dfxm-forward`** — *"one known sample, scan the instrument."* You give one
  crystal config; it scans the instrument axes ($\phi, \chi, 2\Delta\theta, z$)
  and returns the image stack (the mosaicity-map / IUCrJ simulation). No
  per-defect labels beyond what you fed in.

- **`dfxm-identify`** — *"vary the defect, produce a **labeled dataset**."* It
  loops over defect configurations and records each one's parameters as ground
  truth. Three modes:

| mode | sweeps | per image | labels |
|---|---|---|---|
| `single` | deterministic: 4 slip planes × 6 Burgers × 36 rotations = **864** (840 after dropping $g\cdot b\approx0$) | 1 dislocation | exact → clean **test set** (Borgi 2025) |
| `multi` | Monte-Carlo: **N random dislocations** (default 2) per image | N dislocations | per-instance ground truth → the **ML training generator** |
| `z-scan` | adds a **depth** axis (z-layers × rocking grid) | 1–2 | 4D stacks |

The rest of this notebook drives **`multi`** mode.

## 3 · Setup & the resolution kernel

On a fresh machine:

```bash
pip install dfxm-geo          # from PyPI  (or `pip install -e .` in the repo)
```

Before any image can be rendered, the pipeline needs a **reciprocal-space
resolution kernel** — a Monte-Carlo sampling of the instrument resolution
function that every forward/identify step convolves with. You build it once with
`dfxm-bootstrap`; the CLIs auto-discover it afterwards.

```bash
dfxm-bootstrap --config demo_multi.toml      # reads the [reciprocal] block
```

The `[reciprocal] Nrays` key sets quality vs. build time:

| `Nrays` | Build time (laptop) | Use |
|---|---|---|
| `2_000_000` | ~2 s | Tutorial / quick exploration |
| `20_000_000` | ~18 s | Development |
| `100_000_000` | ~1 min | Production |

The demo config uses `Nrays = 2_000_000` — fast enough for a walkthrough.
For publication-quality images, raise it to `100_000_000`.

The cell below runs the bootstrap as a subprocess (so its transient memory
is released before we render), writing the kernel into a local `kernel/`
folder next to the notebook so the tutorial is fully self-contained —
nothing system-wide is touched.

In [ ]:
%matplotlib inline
import subprocess
import sys
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import PowerNorm

# --- Preflight: verify dfxm_geo is installed and print its version -----------
try:
    import dfxm_geo

    print(f"dfxm_geo version: {dfxm_geo.__version__}")
except ImportError as exc:
    raise ImportError(
        "dfxm_geo not found.\n"
        "Install from the repo:  pip install -e .\n"
        "Or from PyPI:           pip install dfxm-geo"
    ) from exc

HERE = Path.cwd()  # run this notebook from its own folder
CONFIG = HERE / "demo_multi.toml"
OUTDIR = HERE / "out_demo"
KERNEL_DIR = HERE / "kernel"
KERNEL_DIR.mkdir(exist_ok=True)
assert CONFIG.exists(), "Run this notebook from examples/identification_ml_tutorial/"

# Each heavy step runs in its own subprocess (a fresh memory budget) pointed at
# this local kernel/ folder, so the tutorial is self-contained and laptop-safe.
# The KERNEL_ENV prefix overrides fm.pkl_fpath so the child process discovers
# the kernel in kernel/ rather than the default reciprocal_space/pkl_files/.
KERNEL_ENV = "import dfxm_geo.direct_space.forward_model as fm; fm.pkl_fpath = %r; " % (
    KERNEL_DIR.resolve().as_posix() + "/"
)
print("config :", CONFIG.name)
print("kernel :", KERNEL_DIR.name + "/")
print("output :", OUTDIR.name + "/")

In [ ]:
# Build the resolution kernel — reciprocal-space Monte Carlo. Equivalent to:
#     dfxm-bootstrap --config demo_multi.toml --output kernel/...tutorial.npz
# Run as a subprocess so its transient memory is fully released before we
# start rendering images.

KERNEL_FILE = KERNEL_DIR / "Resq_i_h-1_k1_l-1_17keV_tutorial.npz"

if KERNEL_FILE.exists():
    print("reusing kernel:", KERNEL_FILE.name)
else:
    t0 = time.perf_counter()
    res = subprocess.run(
        [
            sys.executable,
            "-c",
            "from dfxm_geo.reciprocal_space.kernel import cli_main; raise SystemExit(cli_main())",
            "--config",
            str(CONFIG),
            "--output",
            str(KERNEL_FILE),
        ],
        capture_output=True,
        text=True,
    )
    if res.returncode != 0:
        print("STDERR:", res.stderr[-2000:])
    assert res.returncode == 0, "dfxm-bootstrap failed — see stderr above"
    print(f"--> kernel built in {time.perf_counter() - t0:.0f} s")
    print(res.stdout)

## 4 · Generate a tiny labeled dataset

We'll make **8 images**, each a fresh scene of **2 random dislocations**. The
config below is the whole specification. Note two ML-relevant switches:

- `render_per_dislocation = true` — also write each dislocation rendered *alone*
  and *noiseless* → clean per-instance labels (segmentation / detection).
- `write_strain_provenance = false` — drop the bulky per-ray strain dump from
  the HDF5. This is the storage lever for big datasets (more in §8).

In [ ]:
# The full data specification — one TOML file:
print(CONFIG.read_text(encoding="utf-8"))

In [ ]:
# Generate. Equivalent to the production command:
#     dfxm-identify --config demo_multi.toml --output out_demo --mode multi
# Run in its own subprocess (fresh memory) pointed at the local kernel/ we built.
# Idempotent: delete out_demo/ to force a regenerate.
if (OUTDIR / "dfxm_identify.h5").exists():
    print(f"reusing existing dataset in {OUTDIR.name}/  (delete it to regenerate)")
else:
    res = subprocess.run(
        [
            sys.executable,
            "-c",
            KERNEL_ENV
            + "from dfxm_geo.pipeline import cli_main_identify; raise SystemExit(cli_main_identify())",
            "--config",
            str(CONFIG),
            "--output",
            str(OUTDIR),
            "--mode",
            "multi",
        ],
        capture_output=True,
        text=True,
    )
    print(res.stdout)
    assert res.returncode == 0, res.stderr[-1500:]

## 5 · What got written

A small **master** file (`dfxm_identify.h5`) plus one `scanNNNN/` folder per
image. The master is the BLISS-style index: each image is a scan entry
`/N.1/` carrying its labels under `/N.1/sample/dislocations/{0,1}/` and an
**external link** to its detector image. Per-scan folders hold the actual
detector HDF5s (combined + the two noiseless per-instance renders).

Look at the sizes: with `write_strain_provenance = false`, the master is a few
hundred KB and each detector is ~15 KB (the images are sparse weak-beam contrast
→ they compress hard).

In [ ]:
master = OUTDIR / "dfxm_identify.h5"
print(f"master  {master.name:<22} {master.stat().st_size / 1024:7.0f} KB")
for scan in sorted(OUTDIR.glob("scan*"))[:3]:
    for f in sorted(scan.glob("*.h5")):
        print(f"  {scan.name}/{f.name:<32} {f.stat().st_size / 1024:7.0f} KB")
print("  ... (8 scan folders total)")

## 6 · The labels your model trains on

Everything needed to supervise a model lives under `/N.1/sample/dislocations/`.
For each dislocation:

| field | type | example ML target |
|---|---|---|
| `slip_plane_normal` | int `[h,k,l]` | **classification** (4 {111} planes) |
| `burgers` | int `[h,k,l]` | **classification** (6 Burgers vectors) / screw–edge–mixed character |
| `rotation_deg` | float | **regression** (line-direction angle) |
| `position_um` | float `(x,y)` | **regression / detection** (instance location) |

Plus the per-instance noiseless renders (`_dis0`, `_dis1`) for
**instance segmentation / counting**.

In [ ]:
def read_labels(master_path):
    rows = []
    with h5py.File(master_path, "r") as f:
        scan_ids = sorted((k for k in f if k.endswith(".1")), key=lambda s: int(s.split(".")[0]))
        for sid in scan_ids:
            dgrp = f[f"{sid}/sample/dislocations"]
            for di in sorted(dgrp, key=int):
                g = dgrp[di]
                rows.append(
                    (
                        int(sid.split(".")[0]),
                        int(di),
                        tuple(int(x) for x in g["slip_plane_normal"][()]),
                        tuple(int(x) for x in g["burgers"][()]),
                        round(float(g["rotation_deg"][()]), 1),
                        tuple(round(float(x), 2) for x in g["position_um"][()][:2]),
                    )
                )
    return rows


rows = read_labels(master)
hdr = f"{'img':>3} {'disl':>4}  {'slip_plane':>12} {'burgers':>10} {'rot_deg':>7} {'pos_um (x,y)':>16}"
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(f"{r[0]:>3} {r[1]:>4}  {str(r[2]):>12} {str(r[3]):>10} {r[4]:>7} {str(r[5]):>16}")

## 7 · The images

The detector image is `(1, 510, 170)` float32 — a single frame, 510×170 (the
real DFXM detector aspect; the 1/sin 2θ foreshortening gives the 3:1 ratio).
The **combined** image is what a detector would see (both dislocations + Poisson
noise + intensity scaling); the **per-instance** renders are the same scene with
one dislocation present and **no noise** — i.e. pixel-accurate labels.

Contrast is sparse and thread-like: at a fixed rocking angle only the parts of
each strain field that satisfy the diffraction condition light up (weak-beam-like).
Each panel below is normalised to its own peak (so the unscaled per-instance
renders are visible next to the ×7-scaled combined); compare **shape and
position** — the combined panel is the superposition of the two instance panels.

In [ ]:
def read_image(master_path, sample, det="dfxm_sim_detector"):
    """Read a detector frame from the master HDF5 file.

    Follows the external link from /N.1/instrument/<det>/data to the
    per-scan detector file. Falls back to opening the per-scan file directly
    if the external link cannot be resolved (e.g. paths differ across
    machines, or the master was copied without the scan folders).
    """
    master_path = Path(master_path)
    with h5py.File(master_path, "r") as f:
        try:
            return np.squeeze(f[f"{sample}.1/instrument/{det}/data"][()])
        except (KeyError, OSError, ValueError):
            pass
    # Fallback: open the per-scan file directly.
    # The per-scan detector file uses the LIMA naming convention:
    #   out_demo/scan{N:04d}/dfxm_sim_detector_0000.h5  (combined)
    #   out_demo/scan{N:04d}/dfxm_sim_detector_dis0_0000.h5  (per-instance)
    suffix_map = {
        "dfxm_sim_detector": "dfxm_sim_detector_0000.h5",
        "dfxm_sim_detector_dis0": "dfxm_sim_detector_dis0_0000.h5",
        "dfxm_sim_detector_dis1": "dfxm_sim_detector_dis1_0000.h5",
    }
    scan_file = master_path.parent / f"scan{sample:04d}" / suffix_map[det]
    with h5py.File(scan_file, "r") as f:
        # LIMA layout: /entry_0000/measurement/data or /1.1/instrument/<det>/data
        for path in (
            "entry_0000/measurement/data",
            f"1.1/instrument/{det}/data",
        ):
            if path in f:
                return np.squeeze(f[path][()])
    raise KeyError(f"Could not find detector data for scan {sample} / {det}")


n_show = 8
fig, axes = plt.subplots(n_show, 3, figsize=(6.5, 2.5 * n_show))
titles = ["combined  (detector + noise)", "dislocation 0  (label)", "dislocation 1  (label)"]
dets = ["dfxm_sim_detector", "dfxm_sim_detector_dis0", "dfxm_sim_detector_dis1"]
for row in range(n_show):
    s = row + 1
    imgs = [read_image(master, s, d) for d in dets]
    for col, (ax, im) in enumerate(zip(axes[row], imgs, strict=True)):
        # normalise each panel to its own peak so the noiseless per-instance
        # renders (unscaled) are as visible as the x7-scaled combined image
        ax.imshow(
            im, cmap="magma", norm=PowerNorm(0.45, vmin=0, vmax=im.max() or 1.0), aspect="auto"
        )
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(titles[col], fontsize=8)
    axes[row, 0].set_ylabel(f"image {s}", fontsize=9)
fig.suptitle(
    "multi mode: 2 random dislocations / image, with per-instance ground truth", fontsize=10
)
fig.tight_layout()
fig.savefig(HERE / "preview.png", dpi=110, bbox_inches="tight")
plt.show()

## 8 · Scaling to 100k+ images

Two paths to a 100k labeled dataset:

**Path A — `dfxm-identify --mode multi` (clean per-instance labels).**
Raise `n_samples` and shard across seeds for independent, reproducible chunks.
The `--seed INT` flag overrides `noise.rng_seed` at the command level, so one
TOML config can drive an entire array job without editing files:

```bash
# One shard on a workstation:
dfxm-identify --config demo_multi.toml --output out_shard_42 --seed 42

# In-node fan-out (many configs in parallel on one big node):
python scripts/fanout.py --manifest sweep_configs/ \
    --output /scratch/run --mode identify --n-workers 8

# On the DTU LSF cluster (each array task gets a distinct seed):
bsub < lsf/identify_array.bsub
```

The cluster template (`lsf/identify_array.bsub`) runs a `[1-100]` array job —
each task calls `dfxm-identify --seed "${LSB_JOBINDEX}"`, so 100 tasks × the
per-task `n_samples` = the total dataset size. Adjust `#BSUB -J` and
`n_samples` together.

The cell below writes seed-sharded configs (as an alternative to the `--seed`
flag; either approach works).

In [ ]:
# Illustrative ONLY — writes seed-sharded configs; does not run them.
# These are equivalent to passing `--seed k` on the command line.
import re

shard_dir = HERE / "sweep_configs"
shard_dir.mkdir(exist_ok=True)
N_SHARDS, SAMPLES_PER_SHARD = 8, 12_500  # 8 x 12_500 = 100_000 images
base = CONFIG.read_text(encoding="utf-8")
for k in range(N_SHARDS):
    txt = re.sub(r"rng_seed\s*=\s*\d+", f"rng_seed = {k}", base)
    txt = re.sub(r"n_samples\s*=\s*\d+", f"n_samples = {SAMPLES_PER_SHARD}", txt)
    (shard_dir / f"shard_{k:02d}.toml").write_text(txt, encoding="utf-8")
print(f"wrote {N_SHARDS} seed-sharded configs -> {N_SHARDS * SAMPLES_PER_SHARD:,} images total")
print(sorted(p.name for p in shard_dir.glob("*.toml")))

**Path B — `dfxm-forward` + `random_dislocations` (proven throughput, no per-instance labels).**
This path fans out via `scripts/fanout.py` (config-level fan-out — ~8 configs ×
16 threads beats one over-threaded scan, which plateaus ~16 cores / memory-bandwidth bound):

```bash
# one node, many configs in parallel (forward mode — default):
python scripts/fanout.py --manifest configs/sweep/ --output /scratch/run \
       --n-workers 8 --threads-per-worker 16
# or the identify equivalent (mode=identify):
python scripts/fanout.py --manifest sweep_configs/ --output /scratch/run \
       --mode identify --n-workers 8
# on the DTU LSF cluster:
bsub < lsf/fanout.bsub    # set MODE=identify in the bsub header to fan out identify
```

Benchmarked envelope for 100k images (forward path): **~3 node-hours**, and the
storage lever matters a lot —

| `write_strain_provenance` | per config | 100k dataset |
|---|---|---|
| `true` (default) | ~106 MB | ~0.51 TB |
| **`false`** (ML/batch, set in demo_multi.toml) | ~32 KB | **~3 GB** |

The trade-off: Path A gives you clean per-instance labels (segmentation / detection);
Path B gives you population/density fields but has more cluster profiling behind it.
For a mixed strategy, run Path A via the identify fan-out for labeled data and Path B
for unlabeled augmentation.

## 9 · What exists vs. what I could add

Where the pipeline is **now**:

- ✅ `multi` mode: exactly 2 random dislocations/image, full per-instance labels
- ✅ noiseless per-instance renders (`render_per_dislocation`)
- ✅ HDF5/BLISS output (silx / darfix / darling readable), float32 detector
- ✅ deterministic, seed-reproducible; storage lever for big runs
- ✅ Poisson noise + intensity scaling for detector realism
- ✅ identify fan-out launcher: `--seed INT` flag + `fanout.py --mode identify` + LSF array job

Things I can build — **tell me which matter for your models:**

| idea | what it would give you | rough size |
|---|---|---|
| **N > 2 dislocations / image** | denser scenes, variable count (detection/counting) | small |
| **pixel segmentation masks** | turn per-instance renders into integer instance masks | small |
| **regression on the strain field** | dense $H_g$ map as a target (`write_strain_provenance=true`) | already there, needs a loader |
| **more slip systems** | full 12 FCC slip systems (today samples a subset) | one-liner |
| **multi-reflection per sample** | several $hkl$ per scene → richer multi-channel input | medium (oblique arc exists) |
| **arbitrary lattices via `.cif`** | beyond FCC Al (other materials) | larger (v3.0.0) |
| **detector realism** | background, PSF, dead pixels, dose sweeps | small–medium |
| **3D / z-scan datasets** | depth-resolved stacks (`z-scan` mode) | exists, needs label plumbing |

**Questions for you:** classification, detection/segmentation, or regression
first? How many dislocations per image? Do you want the dense strain field as a
target, or just the categorical/geometric labels? One reflection or several?

## 10 · Recap

- `dfxm-identify --mode multi` turns a TOML config into images **with labels** —
  the ML-data generator.
- Labels live under `/N.1/sample/dislocations/`; per-instance noiseless renders
  give clean segmentation/detection targets.
- **Scale via the built-in fan-out:** `dfxm-identify --seed INT` for independent
  shards on a workstation; `python scripts/fanout.py --mode identify` for in-node
  parallelism; `bsub < lsf/identify_array.bsub` for LSF array jobs on the cluster
  (each task gets a distinct `--seed ${LSB_JOBINDEX}`).
- Flip `write_strain_provenance=false` to keep 100k images around ~3 GB.

**Run this notebook:** from `examples/identification_ml_tutorial/`, with the
`dfxm-geo` environment active. The generation cell takes ~45 s on a laptop
(numba warmup + 8 images; subsequent runs reuse the kernel and skip the bootstrap).